In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.split import make_time_folds, expanding_window_splits
from src.features.build_features import build_features

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PREDICTIONS = PROJECT_ROOT / "data" / "predictions"
REPORTS_FIGURES = PROJECT_ROOT / "reports" / "figures"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (12, 6)

# Загружаем данные
train = pd.read_parquet(DATA_INTERIM / "train.parquet", engine="pyarrow")
test = pd.read_parquet(DATA_INTERIM / "test.parquet", engine="pyarrow")
rename_map = {c: c.replace("id-", "id_") for c in test.columns if c.startswith("id-")}
test = test.rename(columns=rename_map)

# FE pipeline
folds = make_time_folds(train["TransactionDT"], n_splits=5)
target = train["isFraud"].astype(np.int8)

train_fe, test_fe = build_features(
    train=train.copy(),
    test=test.copy(),
    target=target,
    folds=folds,
    verbose=False,  # молчать — мы это уже видели
)

# Подготовка X / y
drop_cols = ["TransactionID", "isFraud", "TransactionDT", "uid"]
y = train_fe["isFraud"].astype(np.int8)
X = train_fe.drop(columns=drop_cols)
cat_cols = X.select_dtypes(include=["category"]).columns.tolist()

print(f"Train FE: {train_fe.shape}")
print(f"X: {X.shape}, cat cols: {len(cat_cols)}")

In [ ]:
# Используем только fold 4 для feature selection
# Это train на folds 0-3, validation на fold 4
splits = expanding_window_splits(folds, n_splits=5)
train_idx, valid_idx = splits[-1]  # последний фолд

X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
X_va, y_va = X.iloc[valid_idx], y.iloc[valid_idx]

print(f"Train size: {len(X_tr):,}")
print(f"Valid size: {len(X_va):,}")

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
    "n_jobs": -1,
    "random_state": 42,
}

t0 = time.time()
train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
valid_data = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols,
                          reference=train_data)

model_fs = lgb.train(
    params=lgb_params,
    train_set=train_data,
    num_boost_round=2000,
    valid_sets=[valid_data],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=False),
        lgb.log_evaluation(period=0),
    ],
)

preds_va = model_fs.predict(X_va, num_iteration=model_fs.best_iteration)
auc_va = roc_auc_score(y_va, preds_va)
print(f"\nFold 4 AUC: {auc_va:.5f} (best_iter={model_fs.best_iteration})")
print(f"Time: {time.time() - t0:.0f}s")

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, ClassifierMixin

print("Computing permutation importance...")
t0 = time.time()

class LGBWrapper(ClassifierMixin, BaseEstimator):
    _estimator_type = "classifier"

    def __init__(self, model=None, feature_names=None):
        self.model = model
        self.feature_names = feature_names

    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        self.feature_names_in_ = np.array(self.feature_names)
        return self

    def predict(self, X):
        preds = self.model.predict(X, num_iteration=self.model.best_iteration)
        return (preds > 0.5).astype(int)

    def predict_proba(self, X):
        preds = self.model.predict(X, num_iteration=self.model.best_iteration)
        return np.column_stack([1 - preds, preds])

wrapper = LGBWrapper(model=model_fs, feature_names=X.columns.tolist())
wrapper.fit(X_va.iloc[:10], y_va.iloc[:10])  # чтобы выставить classes_

result = permutation_importance(
    wrapper,
    X_va,
    y_va,
    n_repeats=3,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
)

elapsed = time.time() - t0
print(f"Done in {elapsed:.0f}s")

perm_imp = pd.DataFrame({
    "feature": X.columns,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

print(f"\nТоп-20 фичей:")
print(perm_imp.head(20).to_string(index=False))
print()
print(f"Фичей с importance <= 0: {(perm_imp['importance_mean'] <= 0).sum()}")
print(f"Фичей с importance < -0.001: {(perm_imp['importance_mean'] < -0.001).sum()}")

In [ ]:
# Фичи, которые оставляем
keep_mask = perm_imp["importance_mean"] > 0
features_to_keep = perm_imp.loc[keep_mask, "feature"].tolist()

print(f"Всего фичей:     {len(perm_imp)}")
print(f"Оставляем:       {len(features_to_keep)}")
print(f"Удаляем:         {(~keep_mask).sum()}")

# Визуально: распределение importance
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(perm_imp["importance_mean"], bins=100, color="steelblue")
ax.axvline(0, color="red", linestyle="--", label="importance = 0")
ax.set_xlabel("Permutation importance (mean ROC-AUC drop)")
ax.set_ylabel("Number of features")
ax.set_title("Feature importance distribution")
ax.legend()
ax.set_yscale("log")
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "07_permutation_importance_hist.png", bbox_inches="tight")
plt.show()

# Сохраняем выбранные фичи в файл — пригодится для воспроизведения
import json
features_path = PROJECT_ROOT / "reports" / "selected_features_v4.json"
with open(features_path, "w") as f:
    json.dump({
        "features_to_keep": features_to_keep,
        "features_dropped": perm_imp.loc[~keep_mask, "feature"].tolist(),
        "criterion": "permutation_importance_mean > 0",
        "source_auc": float(auc_va),
    }, f, indent=2)
print(f"\nSaved: {features_path.name}")

In [ ]:
# Новый X только с отобранными фичами
X_selected = X[features_to_keep]
cat_cols_selected = [c for c in cat_cols if c in features_to_keep]
print(f"X_selected: {X_selected.shape}")
print(f"Cat cols: {len(cat_cols_selected)} (из {len(cat_cols)})")

# Полный CV
splits = expanding_window_splits(folds, n_splits=5)
oof_preds_v4 = np.zeros(len(X_selected), dtype=np.float32)
cv_scores_v4 = []
models_v4 = []

for fold_num, (train_idx, valid_idx) in enumerate(splits, start=1):
    t0 = time.time()
    X_tr = X_selected.iloc[train_idx]
    X_va = X_selected.iloc[valid_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[valid_idx]

    train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols_selected)
    valid_data = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols_selected,
                              reference=train_data)

    model = lgb.train(
        params=lgb_params,
        train_set=train_data,
        num_boost_round=2000,
        valid_sets=[valid_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    oof_preds_v4[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    fold_auc = roc_auc_score(y_va, oof_preds_v4[valid_idx])
    cv_scores_v4.append(fold_auc)
    models_v4.append(model)

    elapsed = time.time() - t0
    print(f"Fold {fold_num}: AUC = {fold_auc:.5f} "
          f"(best_iter={model.best_iteration}, time={elapsed:.0f}s)")

print()
print(f"CV AUC: {np.mean(cv_scores_v4):.5f} ± {np.std(cv_scores_v4):.5f}")
valid_mask = folds > 0
print(f"OOF AUC: {roc_auc_score(y[valid_mask], oof_preds_v4[valid_mask]):.5f}")

print()
print(f"v3 (471 features):  CV 0.92907 ± 0.02600")
print(f"v4 ({len(features_to_keep)} features):  CV {np.mean(cv_scores_v4):.5f} "
      f"± {np.std(cv_scores_v4):.5f}")
print(f"Delta CV:           {np.mean(cv_scores_v4) - 0.92907:+.5f}")


In [ ]:
# Test prediction с отобранными фичами
X_test = test_fe.drop(columns=["TransactionID", "TransactionDT", "uid"])
X_test_selected = X_test[features_to_keep].copy()

for col in cat_cols_selected:
    if col in X_test_selected.columns:
        X_test_selected[col] = X_test_selected[col].astype(X_selected[col].dtype)

test_preds = np.zeros(len(X_test_selected), dtype=np.float32)
for model in models_v4:
    p = model.predict(X_test_selected, num_iteration=model.best_iteration)
    test_preds += p / len(models_v4)

print(f"Predictions: min={test_preds.min():.4f}, max={test_preds.max():.4f}, "
      f"mean={test_preds.mean():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_fe["TransactionID"],
    "isFraud": test_preds,
})
submission.to_csv(DATA_PREDICTIONS / "sub_lgbm_fs_v4.csv", index=False)

oof_df = pd.DataFrame({
    "TransactionID": train_fe["TransactionID"],
    "isFraud_true": y,
    "isFraud_pred": oof_preds_v4,
    "fold": folds,
})
oof_df.to_csv(DATA_PREDICTIONS / "oof_lgbm_fs_v4.csv", index=False)
print("Saved: sub_lgbm_fs_v4.csv + oof_lgbm_fs_v4.csv")